# 04 · Data-scaling run (blueprint §7.3)

One training run per notebook, set in the parameters cell below. It trains on the first `N_TRAIN` examples of the nested,
seed-42 ordering of the 6,218 cleaned ScienceQA image questions, so smaller budgets are subsets of larger ones. Everything
else matches the pilots: A1 targets, r=16, α=32, 2 epochs, LR 1e-4, batch 1 × 8.

It then scores the adapter on the **full validation split (2,097 questions)**, and optionally M0 too.
Validation data only; the test split stays sealed.

Settings: Accelerator → GPU T4, Internet → On.

In [ ]:
# Parameters
METHOD = "qlora"          # "qlora" or "lora"
N_TRAIN = 2000          # 500 / 2000 / 5000 / 6218 (= all)
EVAL_BASE = True      # also score the base model M0 on the full validation split
SEED = 42

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess, sys

REPO_DIR = "/tmp/VisionTune"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/akhilnathreddy/VisionTune.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
COMMIT = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
print("inputs mounted:", subprocess.run("find /kaggle/input -maxdepth 3 | head -20", shell=True,
                                        capture_output=True, text=True).stdout or "none")

In [ ]:
!pip install -q -U "transformers>=4.57" "peft>=0.17" "accelerate>=1.0" "datasets>=3.0" "bitsandbytes>=0.45" qwen-vl-utils
!pip install -q -e .

In [ ]:
import json, shutil, time
from pathlib import Path

RUN_NAME = f"scienceqa_qwen25vl3b_{METHOD}_r16_n{N_TRAIN}_seed{SEED}"
OUT = Path("/kaggle/working") / RUN_NAME
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
timings = {}

def sh(cmd):
    print("$", cmd, flush=True)
    t0 = time.perf_counter()
    subprocess.run(cmd, shell=True, check=True)
    return round(time.perf_counter() - t0, 1)

timings["prepare_s"] = sh("python -m visiontune.data.prepare --config configs/base.yaml --skip-ai2d > /dev/null")

## Train

In [ ]:
overrides = (f"run_name={RUN_NAME} seed={SEED} train.max_train_samples={N_TRAIN} "
             f"train.eval_steps=250 train.save_steps=250 train.save_total_limit=2 train.output_root={OUT}/outputs")
timings["train_s"] = sh(f"python -m visiontune.training.train --config configs/{METHOD}.yaml {overrides}")
run_dir = OUT / "outputs" / RUN_NAME
summary = json.loads((run_dir / "run_summary.json").read_text())
print(json.dumps({k: summary[k] for k in ("train_examples", "train_seconds_this_session", "peak_vram_gb",
                                          "adapter_size_mb", "lora_targets", "final_eval")}, indent=1))
print("eval loss:", [(h["step"], round(h["eval_loss"], 4)) for h in summary["log_history"] if "eval_loss" in h])

## Evaluate on the full validation split

In [ ]:
pred = OUT / "predictions" / f"{RUN_NAME}_validation_full.jsonl"
timings["eval_s"] = sh(f"python -m visiontune.inference.predict --config configs/{METHOD}.yaml "
                       f"--adapter {run_dir}/adapter --split validation --out {pred}")
if EVAL_BASE:
    base_pred = OUT / "predictions" / "base_validation_full.jsonl"
    timings["base_eval_s"] = sh(f"python -m visiontune.inference.predict --config configs/base.yaml "
                                f"--split validation --out {base_pred}")
m = json.loads(pred.with_suffix(".metrics.json").read_text())
result = {"run_name": RUN_NAME, "method": METHOD, "n_train": N_TRAIN, "commit": COMMIT, "timings_s": timings,
          "validation": {k: m.get(k) for k in ("overall", "format", "by_subject", "by_near_dup_in_train",
                                               "latency_s_per_example_batched", "mean_new_tokens",
                                               "generated_tokens_per_s_batched", "peak_vram_gb")},
          "train": {k: summary[k] for k in ("train_seconds_this_session", "peak_vram_gb", "adapter_size_mb",
                                            "lora_targets", "parameters", "final_eval")}}
(OUT / "result.json").write_text(json.dumps(result, indent=2))
print(json.dumps({"acc": m["overall"], "format": m["format"], "near_dup": m["by_near_dup_in_train"]}, indent=1))

In [ ]:
# Keep the adapter, summaries and predictions; drop resumable checkpoints (optimizer state).
for ckpt in (OUT / "outputs").glob("*/checkpoint-*"):
    shutil.rmtree(ckpt)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(OUT)}")